In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from itertools import combinations

final_labels = pd.read_csv("final_labels.csv")

In [2]:
# Count females (label == 1) per model
female_counts = final_labels.groupby("model")["label"].sum()

# Total samples per model
total_counts = final_labels.groupby("model")["label"].count()

female_counts, total_counts


(model
 dalle               29
 midjourney          11
 stable_diffusion    29
 Name: label, dtype: int64,
 model
 dalle               105
 midjourney          101
 stable_diffusion     77
 Name: label, dtype: int64)

In [3]:
models = female_counts.index
results = []

for m1, m2 in combinations(models, 2):
    
    count = np.array([female_counts[m1], female_counts[m2]])
    nobs = np.array([total_counts[m1], total_counts[m2]])
    
    z_stat, p_value = proportions_ztest(count, nobs)
    
    results.append({
        "Model 1": m1,
        "Model 2": m2,
        "z-stat": z_stat,
        "p-value": p_value
    })

results_df = pd.DataFrame(results)
results_df


,Model 1,Model 2,z-stat,p-value
0,dalle,midjourney,3.034225,0.002412
1,dalle,stable_diffusion,-1.436570,0.150840
2,midjourney,stable_diffusion,-4.239505,0.000022


In [4]:
alpha = 0.05
num_tests = len(results_df)
bonf_alpha = alpha / num_tests

results_df["Significant (Bonferroni)"] = results_df["p-value"] < bonf_alpha

results_df

,Model 1,Model 2,z-stat,p-value,Significant (Bonferroni)
0,dalle,midjourney,3.034225,0.002412,True
1,dalle,stable_diffusion,-1.436570,0.150840,False
2,midjourney,stable_diffusion,-4.239505,0.000022,True


In [5]:
female_prop = final_labels.groupby("model")["label"].mean()

diff_results = []

for m1, m2 in combinations(female_prop.index, 2):
    
    diff = female_prop[m1] - female_prop[m2]
    
    diff_results.append({
        "Model 1": m1,
        "Model 2": m2,
        "Difference (p1 - p2)": diff
    })

diff_df = pd.DataFrame(diff_results)
diff_df

,Model 1,Model 2,Difference (p1 - p2)
0,dalle,midjourney,0.167280
1,dalle,stable_diffusion,-0.100433
2,midjourney,stable_diffusion,-0.267712
